In [1]:
import pandas as pd

models = ['claude3', 'google', 'gpt35']
df = {}

for model in models:
    df[model] = pd.read_csv(f"/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/dictionary_collection/growing_dict/final_terms_{model}.csv")
    

In [7]:
# Align the dataframes based on the "English" column using an inner join
df_merged = df['claude3'].merge(df['google'], on="English", suffixes=('_claude', '_google'))\
                      .merge(df['gpt35'], on="English", suffixes=('', '_gpt35'))

# Initialize results
results = {"agreement ratio over three": [], "agreement ratio over two": []}
languages = ["Arabic", "Chinese", "French", "Japanese", "Russian"]

# Calculate agreement ratios
for lang in languages:
    # Columns for comparison
    col_claude = f"{lang}_claude"
    col_google = f"{lang}_google"
    col_gpt35 = f"{lang}"
    
    # Total number of terms after alignment
    total_terms = len(df_merged)
    
    # Calculate agreements
    agree_three = sum(
        (df_merged[col_claude] == df_merged[col_google]) & (df_merged[col_google] == df_merged[col_gpt35])
    )
    agree_two = sum(
        (df_merged[col_claude] == df_merged[col_google]) | 
        (df_merged[col_google] == df_merged[col_gpt35]) | 
        (df_merged[col_claude] == df_merged[col_gpt35])
    ) - agree_three  # Subtract three-way agreement
    
    # Calculate ratios
    results["agreement ratio over three"].append(agree_three / total_terms if total_terms > 0 else 0)
    results["agreement ratio over two"].append(agree_two / total_terms if total_terms > 0 else 0)

# Create a results dataframe
agreement_df_aligned = pd.DataFrame(results, index=languages) * 100

agreement_df_aligned


,agreement ratio over three,agreement ratio over two
Arabic,10.110216,30.521407
Chinese,42.708775,36.816448
French,9.855871,45.146248
Japanese,16.596015,40.440865
Russian,17.931327,38.363713


In [9]:
print(agreement_df_aligned.T.to_latex(float_format="%.2f%%"))

\begin{tabular}{lrrrrr}
\toprule
 & Arabic & Chinese & French & Japanese & Russian \\
\midrule
agreement ratio over three & 10.11% & 42.71% & 9.86% & 16.60% & 17.93% \\
agreement ratio over two & 30.52% & 36.82% & 45.15% & 40.44% & 38.36% \\
\bottomrule
\end{tabular}

